In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.transforms import trainval_transforms, revert_normalization, revert_standardization
from src.dataset import ImageDataset
import torch

annot_path = Path("../data/preprocessed/trainval/annotations.csv")
img_dir = Path("../data/preprocessed/trainval/images")

# test dataset without transforms
dataset = ImageDataset(annot_path, img_dir, transform=trainval_transforms)

In [3]:
from src.model import Model
from torch.utils.data import DataLoader

trainval_dl = DataLoader(dataset, 8, True)
X_batch, y_batch = next(iter(trainval_dl))
model = Model()

model.eval()
with torch.no_grad():
    preds_batch = model(X_batch)
    
preds_batch.shape, preds_batch

(torch.Size([8, 7, 7, 25]),
 tensor([[[[ 3.8421e-03, -8.4367e-03,  5.0099e-03,  ...,  1.2467e-03,
             6.6599e-03,  7.5470e-03],
           [ 3.0784e-03, -1.0447e-02, -1.2652e-02,  ...,  1.4813e-02,
             2.0088e-02, -1.4607e-02],
           [-1.6871e-02, -1.5597e-02, -9.0629e-03,  ..., -5.5713e-03,
            -3.7986e-04, -1.1771e-02],
           ...,
           [ 1.4774e-02,  1.9334e-02,  5.9696e-03,  ...,  9.3003e-04,
            -7.8948e-03, -7.1425e-03],
           [-8.2755e-03, -1.7106e-02, -1.5571e-02,  ...,  9.0817e-03,
            -3.4015e-03,  3.2696e-04],
           [ 4.6601e-03,  8.5619e-03, -6.7976e-03,  ..., -1.4386e-02,
            -2.4712e-03,  1.0869e-02]],
 
          [[-2.5219e-03, -5.3053e-03, -6.0647e-03,  ..., -4.8708e-03,
             1.3584e-02, -1.5399e-03],
           [-1.8393e-02,  2.0601e-03, -1.2802e-02,  ..., -1.2401e-03,
            -1.7735e-02, -2.0555e-02],
           [-2.3687e-03, -7.4284e-03,  1.5886e-02,  ...,  3.6399e-03,
           

In [4]:
from src.postprocessing import decode_preds_batch

decoded_preds_batch = decode_preds_batch(preds_batch)
decoded_preds_batch[0]

tensor([[ 9.0000e+00,  1.4773e-04, -5.1424e-01, -1.1573e+00, -2.3498e-01,
          3.3446e-01],
        [ 1.7000e+01, -2.1168e-04,  3.0349e+01, -1.8800e+00,  3.3667e+01,
          2.6197e+00],
        [ 7.0000e+00, -1.6383e-04,  6.4724e+01, -9.1832e-03,  6.3476e+01,
         -9.4272e-02],
        [ 1.4000e+01,  2.4726e-05,  9.3442e+01, -5.5075e-01,  9.7677e+01,
          1.5105e-01],
        [ 1.5000e+01, -1.4972e-04,  1.2815e+02,  6.4232e-01,  1.2836e+02,
         -1.1261e+00],
        [ 1.5000e+01,  6.4091e-06,  1.5835e+02,  2.0375e-01,  1.6038e+02,
         -5.5818e-01],
        [ 8.0000e+00,  1.9784e-04,  1.9356e+02, -2.1364e-01,  1.9034e+02,
         -7.6720e-01],
        [ 6.0000e+00, -2.5612e-05,  6.0274e-01,  3.0939e+01, -4.8832e-01,
          3.3982e+01],
        [ 1.4000e+01, -2.4211e-04,  3.2156e+01,  3.4403e+01,  3.1878e+01,
          3.0430e+01],
        [ 1.1000e+01, -4.9207e-04,  6.3139e+01,  3.2398e+01,  6.3955e+01,
          3.2182e+01],
        [ 1.2000e+01, -3.9569e

In [5]:
from src.postprocessing import filter_and_group_preds

filtered_grouped_preds = filter_and_group_preds(decoded_preds_batch[0])
filtered_grouped_preds

{'cow': [tensor([ 9.0000e+00,  1.4773e-04, -5.1424e-01, -1.1573e+00, -2.3498e-01,
           3.3446e-01]),
  tensor([9.0000e+00, 1.4624e-05, 1.2753e+02, 1.2884e+02, 1.2879e+02, 1.2789e+02])],
 'person': [tensor([ 1.4000e+01,  2.4726e-05,  9.3442e+01, -5.5075e-01,  9.7677e+01,
           1.5105e-01]),
  tensor([ 1.4000e+01,  3.3917e-04, -9.1068e-03,  1.9392e+02, -8.5460e-01,
           1.9062e+02])],
 'pottedplant': [tensor([ 1.5000e+01,  6.4091e-06,  1.5835e+02,  2.0375e-01,  1.6038e+02,
          -5.5818e-01]),
  tensor([1.5000e+01, 2.3660e-04, 6.6582e+01, 1.6042e+02, 6.1576e+01, 1.5850e+02])],
 'chair': [tensor([ 8.0000e+00,  1.9784e-04,  1.9356e+02, -2.1364e-01,  1.9034e+02,
          -7.6720e-01])],
 'dog': [tensor([1.1000e+01, 2.1996e-04, 6.4017e+01, 6.4810e+01, 6.4265e+01, 6.4191e+01]),
  tensor([1.1000e+01, 2.5562e-05, 1.5817e+02, 9.7266e+01, 1.6068e+02, 9.5648e+01]),
  tensor([ 1.1000e+01,  3.3115e-04, -5.0827e-01,  1.5827e+02,  3.5781e-01,
           1.6153e+02]),
  tensor([1.

In [6]:
from src.postprocessing import sort_class_preds_by_confidence

sort_class_preds_by_confidence(filtered_grouped_preds)
filtered_grouped_preds

{'cow': [tensor([9.0000e+00, 1.4624e-05, 1.2753e+02, 1.2884e+02, 1.2879e+02, 1.2789e+02]),
  tensor([ 9.0000e+00,  1.4773e-04, -5.1424e-01, -1.1573e+00, -2.3498e-01,
           3.3446e-01])],
 'person': [tensor([ 1.4000e+01,  2.4726e-05,  9.3442e+01, -5.5075e-01,  9.7677e+01,
           1.5105e-01]),
  tensor([ 1.4000e+01,  3.3917e-04, -9.1068e-03,  1.9392e+02, -8.5460e-01,
           1.9062e+02])],
 'pottedplant': [tensor([ 1.5000e+01,  6.4091e-06,  1.5835e+02,  2.0375e-01,  1.6038e+02,
          -5.5818e-01]),
  tensor([1.5000e+01, 2.3660e-04, 6.6582e+01, 1.6042e+02, 6.1576e+01, 1.5850e+02])],
 'chair': [tensor([ 8.0000e+00,  1.9784e-04,  1.9356e+02, -2.1364e-01,  1.9034e+02,
          -7.6720e-01])],
 'dog': [tensor([1.1000e+01, 2.5562e-05, 1.5817e+02, 9.7266e+01, 1.6068e+02, 9.5648e+01]),
  tensor([1.1000e+01, 1.0744e-04, 1.9047e+02, 1.9170e+02, 1.9402e+02, 1.9204e+02]),
  tensor([1.1000e+01, 2.1996e-04, 6.4017e+01, 6.4810e+01, 6.4265e+01, 6.4191e+01]),
  tensor([ 1.1000e+01,  3.31